# Lab 5.10 &mdash; Plan-and-Execute (vs ReAct)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Let the REAL model draft ALL the steps up front
- Run each step with a deterministic executor, feeding results forward
- Contrast plan-and-execute with step-by-step ReAct

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Planning patterns at a glance](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-10"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**ReAct** (Lab 7) decides one step at a time. **Plan-and-execute** drafts the **whole plan first**, then
runs it &mdash; better for long, structured tasks. Here the **real model** authors a plan (a list of
`tool | input` steps), and your **deterministic executor** runs them, substituting the previous **result**
into the next step. The planning is real and model-driven; the execution is real Python you control. If the
model's plan is malformed, your executor still runs a sensible fallback &mdash; graceful, not a crash.

In [ ]:
# DEMO -- the plan is a list of {tool, input}; RESULT means "the previous step's output"
example = [{"tool": "lookup", "input": "population of metropolis"}, {"tool": "calculator", "input": "RESULT*2"}]
print("a plan looks like:", example)

## Build it
Write `make_plan` (ask the model for `tool | input` lines and keep the valid ones) and `executor` (run each
step, substituting the previous `RESULT`).

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

KNOWLEDGE = {"population of metropolis": "120000"}
def _calc(e):
    try:
        return str(safe_calc(e))
    except Exception:
        return "error: not a numeric expression"
TOOLS = {"lookup": lambda k: KNOWLEDGE.get(str(k).lower().strip(), "unknown"), "calculator": _calc}

def make_plan(goal):
    prompt = ___   # TODO: ask the model to output one "<tool> | <input>" line per step (tools: lookup, calculator; RESULT = prior output). A one-line EXAMPLE in the prompt makes an 8b model follow the format.
    text = llm_text(prompt)                 # the REAL model AUTHORS the plan
    plan = []
    for line in text.splitlines():
        if "|" in line:
            tool, _, inp = line.partition("|")
            tool = tool.strip().lower().lstrip("-0123456789. ")
            if ___:   # TODO: keep the step only if tool is a real tool in TOOLS
                plan.append({"tool": tool, "input": inp.strip()})
    return plan

def executor(plan):
    result, trace = None, []
    for step in plan:
        arg = ___   # TODO: replace RESULT in the input with the previous result (str), else use input as-is
        result = ___   # TODO: run this step's tool on arg
        trace.append((step["tool"], arg, result))
    return result, trace

try:
    print("tools ready:", list(TOOLS))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Let the real model write the plan, then execute it deterministically and read the trace.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        plan = make_plan("twice the population of metropolis")   # REAL model writes the plan
        print("MODEL'S PLAN:", plan)
        if not plan:
            print("(the model didn't produce a parseable plan this time -- using a fallback so you can see the executor)")
            plan = [{"tool": "lookup", "input": "population of metropolis"}, {"tool": "calculator", "input": "RESULT*2"}]
        final, trace = executor(plan)
        print("EXECUTION TRACE:", trace)
        print("FINAL:", final)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The model **front-loaded** the thinking (wrote the whole plan); your executor then ran it step by step, threading `RESULT` through.
- Contrast with **ReAct** (Lab 7), which interleaves think/act &mdash; same tools, different control pattern.
- Real model plans can be messy; parsing only valid `tool` lines + a fallback is what keeps the executor robust.

## Your turn (open task &mdash; no grader)
Change the goal (e.g. *"the population of Metropolis plus 10000"*) and re-run. **What good looks like:**
the model writes a 2-step plan and the executor threads the looked-up number into the arithmetic step. Compare
the plan-first shape here to the interleaved ReAct trace from Lab 7 &mdash; which fits which task?

---
### Key takeaway
Plan-and-execute front-loads the thinking; ReAct interleaves it. A REAL model wrote the plan; your deterministic executor ran it -- same tools, different control pattern.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>